{
  "nbformat": 4,
  "nbformat_minor": 5,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Aula 5 – MLP em Keras: Experimentos, LOSS, Baseline e Overfitting\n",
        "\n",
        "## Objetivos de aprendizado\n",
        "\n",
        "- Revisar a MLP construída na aula passada (Aula 4).\n",
        "- Entender, na prática, o que é **LOSS** (função de perda) e por que ela é importante.\n",
        "- Entender o conceito de **modelo baseline**.\n",
        "- Observar como **neurônios, camadas e epochs** afetam treino e validação.\n",
        "- Identificar sinais de **underfitting** e **overfitting** nos gráficos.\n",
        "\n",
        "## Contexto\n",
        "\n",
        "Você continua como analista de dados da loja online. Seu modelo de MLP prevê se um cliente vai **comprar** (1) ou **não comprar** (0).\n",
        "\n",
        "Hoje, seu chefe quer que você ajuste melhor esse modelo, mas **sem decorar os dados de treino** e mantendo o treinamento em um tempo razoável.\n",
        "\n",
        "---\n",
        "\n",
        "## O que é um **baseline**?\n",
        "\n",
        "Antes de começar a \"tunar\" (ajustar) o modelo, precisamos de um ponto de referência.\n",
        "\n",
        "- **Baseline** é um modelo inicial simples, que serve como **referência mínima**.\n",
        "- A ideia é: *\"Qual o desempenho que eu preciso superar para dizer que melhorei alguma coisa?\"*\n",
        "\n",
        "Exemplos de baseline:\n",
        "- Em classificação: sempre prever a classe mais frequente.\n",
        "- Em redes neurais: uma MLP bem simples com poucos neurônios e poucas camadas.\n",
        "\n",
        "Na aula de hoje, vamos definir uma MLP simples como nosso **modelo baseline** e, a partir dele, fazer experimentos para ver o que melhora ou piora."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Setup inicial\n",
        "\n",
        "Nesta aula vamos:\n",
        "- Gerar um dataset sintético simples de clientes (para manter tudo no mesmo notebook).\n",
        "- Separar em treino / validação / teste.\n",
        "- Padronizar as features.\n",
        "\n",
        "> Se na Aula 4 você usou um dataset real (por exemplo, `clientes.csv`), pode adaptar esta parte trocando o `df` gerado aqui pelo `df` da aula anterior."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "# Imports básicos\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "\n",
        "from sklearn.model_selection import train_test_split\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "\n",
        "from tensorflow import keras\n",
        "from tensorflow.keras import layers\n",
        "\n",
        "sns.set(style=\"whitegrid\")\n",
        "plt.rcParams['figure.figsize'] = (8, 4)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 1.1. Gerando um dataset sintético de clientes\n",
        "\n",
        "Vamos criar um conjunto de dados simples com algumas features:\n",
        "- `tempo_site` (minutos no site)\n",
        "- `paginas_vistas`\n",
        "- `valor_medio_carrinho`\n",
        "- `renda_mensal` (em milhares de reais)\n",
        "\n",
        "E uma variável alvo binária `comprou` (0/1).\n",
        "\n",
        "A regra será **aproximadamente lógica**, mas com ruído (como dados reais), de forma que:\n",
        "- quanto mais tempo no site,\n",
        "- mais páginas vistas,\n",
        "- maior valor médio do carrinho,\n",
        "- e maior renda,\n",
        "\n",
        "maior a probabilidade de **compra**."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "np.random.seed(42)\n",
        "\n",
        "n_samples = 2000\n",
        "\n",
        "tempo_site = np.random.exponential(scale=5, size=n_samples) + 1   # em minutos\n",
        "paginas_vistas = np.random.poisson(lam=5, size=n_samples) + 1\n",
        "valor_medio_carrinho = np.random.gamma(shape=2., scale=50., size=n_samples)\n",
        "renda_mensal = np.random.normal(loc=5, scale=2, size=n_samples)  # em milhares\n",
        "\n",
        "# Regra aproximada para compra (probabilidade)\n",
        "logit = (\n",
        "    0.3 * tempo_site +\n",
        "    0.4 * paginas_vistas +\n",
        "    0.02 * valor_medio_carrinho +\n",
        "    0.1 * renda_mensal -\n",
        "    6.0\n",
        ")\n",
        "prob_compra = 1 / (1 + np.exp(-logit))\n",
        "\n",
        "# Gera 0/1 a partir da probabilidade\n",
        "comprou = (np.random.rand(n_samples) < prob_compra).astype(int)\n",
        "\n",
        "df = pd.DataFrame({\n",
        "    'tempo_site': tempo_site,\n",
        "    'paginas_vistas': paginas_vistas,\n",
        "    'valor_medio_carrinho': valor_medio_carrinho,\n",
        "    'renda_mensal': renda_mensal,\n",
        "    'comprou': comprou\n",
        "})\n",
        "\n",
        "df.head()"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "df['comprou'].value_counts(normalize=True)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "Só para visualizar rapidamente a distribuição de compra vs não compra:"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "sns.countplot(x='comprou', data=df)\n",
        "plt.title('Distribuição da variável alvo (comprou)')\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### 1.2. Separando treino / validação / teste e padronizando\n",
        "\n",
        "- Treino: 60%\n",
        "- Validação: 20%\n",
        "- Teste: 20%\n",
        "\n",
        "Vamos padronizar as features (média 0, desvio 1) para ajudar a MLP a treinar melhor.\n",
        "\n",
        "> **Importante:** o scaler é ajustado **só no treino** e depois aplicado em validação e teste, para não vazar informação."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "X = df.drop('comprou', axis=1).values\n",
        "y = df['comprou'].values\n",
        "\n",
        "X_train, X_temp, y_train, y_temp = train_test_split(\n",
        "    X, y, test_size=0.4, random_state=42, stratify=y\n",
        ")\n",
        "\n",
        "X_val, X_test, y_val, y_test = train_test_split(\n",
        "    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp\n",
        ")\n",
        "\n",
        "scaler = StandardScaler()\n",
        "X_train = scaler.fit_transform(X_train)\n",
        "X_val   = scaler.transform(X_val)\n",
        "X_test  = scaler.transform(X_test)\n",
        "\n",
        "X_train.shape, X_val.shape, X_test.shape"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. LOSS: o que é e como vamos observar\n",
        "\n",
        "Durante o treino, vamos olhar duas coisas principais:\n",
        "- **LOSS de treino**: erro médio nos dados que o modelo está usando para aprender.\n",
        "- **LOSS de validação**: erro médio em dados **novos** (que o modelo não usa para atualizar os pesos).\n",
        "\n",
        "### Intuição rápida de LOSS\n",
        "\n",
        "- Imagine um jogo de dardos:\n",
        "  - O **alvo** é o valor real (0 ou 1: não comprou / comprou).\n",
        "  - O **dardo** é a previsão do modelo (uma probabilidade entre 0 e 1).\n",
        "  - A **LOSS** mede o quão longe, em média, o dardo cai do centro.\n",
        "\n",
        "> Treinar a rede = ajustar os pesos para diminuir essa distância média (a LOSS).\n",
        "\n",
        "### Por que não usamos só accuracy para treinar?\n",
        "\n",
        "- A **accuracy** olha apenas se a previsão final (0 ou 1) bateu com o valor real.\n",
        "- A **LOSS** olha mais fundo: *o quão errado* ou *o quão certo* estamos, usando valores contínuos.\n",
        "- Os algoritmos de otimização (como gradiente descendente) precisam de uma função **contínua** como a LOSS para saber em que direção ajustar os pesos.\n",
        "\n",
        "Vamos criar uma função para plotar as curvas de LOSS e accuracy durante o treino, para visualizar tudo isso."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def plot_history(history, title='Modelo'):\n",
        "    \"\"\"Plota as curvas de LOSS e Accuracy para treino e validação.\"\"\"\n",
        "    hist = history.history\n",
        "    epochs = range(1, len(hist['loss']) + 1)\n",
        "\n",
        "    plt.figure(figsize=(12, 4))\n",
        "\n",
        "    # LOSS\n",
        "    plt.subplot(1, 2, 1)\n",
        "    plt.plot(epochs, hist['loss'], 'b-', label='Treino (loss)')\n",
        "    plt.plot(epochs, hist['val_loss'], 'r--', label='Validação (loss)')\n",
        "    plt.xlabel('Epoch')\n",
        "    plt.ylabel('LOSS')\n",
        "    plt.title(f'{title} - LOSS')\n",
        "    plt.legend()\n",
        "\n",
        "    # ACCURACY\n",
        "    plt.subplot(1, 2, 2)\n",
        "    plt.plot(epochs, hist['accuracy'], 'b-', label='Treino (acc)')\n",
        "    plt.plot(epochs, hist['val_accuracy'], 'r--', label='Validação (acc)')\n",
        "    plt.xlabel('Epoch')\n",
        "    plt.ylabel('Accuracy')\n",
        "    plt.title(f'{title} - Accuracy')\n",
        "    plt.legend()\n",
        "\n",
        "    plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Nosso modelo **baseline** (MLP simples)\n",
        "\n",
        "Vamos definir uma MLP bem simples como **baseline**:\n",
        "- 1 camada oculta com 16 neurônios e ativação **ReLU**.\n",
        "- 1 neurônio de saída com ativação **sigmóide** (para previsão de probabilidade entre 0 e 1).\n",
        "- Otimizador: **Adam**.\n",
        "- Função de LOSS: **`binary_crossentropy`** (clássica para classificação binária).\n",
        "\n",
        "> Lembre: **baseline** é o nosso modelo de referência. Tudo o que fizermos depois deve ser comparado com ele."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def build_model_baseline(input_dim):\n",
        "    \"\"\"Cria o modelo baseline: MLP simples com 1 camada oculta.\"\"\"\n",
        "    model = keras.Sequential([\n",
        "        layers.Dense(16, activation='relu', input_shape=(input_dim,)),\n",
        "        layers.Dense(1, activation='sigmoid')\n",
        "    ])\n",
        "    model.compile(\n",
        "        optimizer='adam',\n",
        "        loss='binary_crossentropy',  # <<< FUNÇÃO DE LOSS\n",
        "        metrics=['accuracy']\n",
        "    )\n",
        "    return model\n",
        "\n",
        "baseline_model = build_model_baseline(X_train.shape[1])\n",
        "\n",
        "history_baseline = baseline_model.fit(\n",
        "    X_train, y_train,\n",
        "    epochs=30,\n",
        "    batch_size=32,\n",
        "    validation_data=(X_val, y_val),\n",
        "    verbose=0\n",
        ")\n",
        "\n",
        "plot_history(history_baseline, title='Baseline (16 neurônios, 1 camada)')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Interpretação: LOSS de treino x LOSS de validação\n",
        "\n",
        "Observe o gráfico de **LOSS**:\n",
        "- A curva **azul** é o erro médio nos **dados de treino**.\n",
        "- A curva **vermelha** é o erro médio nos **dados de validação**.\n",
        "\n",
        "**Responda (mentalmente ou escrevendo abaixo):**\n",
        "1. As duas curvas de LOSS estão descendo? Elas estão relativamente próximas?\n",
        "2. Você vê algum ponto onde a LOSS de validação começa a subir enquanto a de treino continua caindo?\n",
        "3. A partir desse gráfico, você diria que o modelo está **ok**, com **underfitting** ou com **overfitting**?\n",
        "\n",
        "> Dica: **overfitting** aparece quando a perda de treino continua caindo, mas a de validação começa a subir (modelo decorando o treino).\n",
        "\n",
        "Você pode escrever aqui sua interpretação em forma de comentário:"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "# Espaço para você escrever (opcional)\n",
        "comentario_baseline = \"\"\"Escreva aqui, em 1 ou 2 frases, se você acha que o baseline\n",
        "está mais para underfitting, overfitting ou um meio-termo aceitável, com base nas curvas de LOSS.\n",
        "\"\"\"\n",
        "print(comentario_baseline)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Experimento 1 – Aumentando o número de neurônios (Largura)\n",
        "\n",
        "Agora vamos **dobrar** o número de neurônios da camada oculta:\n",
        "- De 16 → 64 neurônios.\n",
        "\n",
        "O que você espera?\n",
        "- Mais neurônios → mais capacidade de aprender padrões complexos.\n",
        "- Mas também → maior risco de **overfitting**.\n",
        "\n",
        "Vamos treinar e comparar com o baseline."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def build_model_wide(input_dim):\n",
        "    \"\"\"Modelo com camada oculta mais larga (mais neurônios).\"\"\"\n",
        "    model = keras.Sequential([\n",
        "        layers.Dense(64, activation='relu', input_shape=(input_dim,)),\n",
        "        layers.Dense(1, activation='sigmoid')\n",
        "    ])\n",
        "    model.compile(\n",
        "        optimizer='adam',\n",
        "        loss='binary_crossentropy',\n",
        "        metrics=['accuracy']\n",
        "    )\n",
        "    return model\n",
        "\n",
        "wide_model = build_model_wide(X_train.shape[1])\n",
        "\n",
        "history_wide = wide_model.fit(\n",
        "    X_train, y_train,\n",
        "    epochs=30,\n",
        "    batch_size=32,\n",
        "    validation_data=(X_val, y_val),\n",
        "    verbose=0\n",
        ")\n",
        "\n",
        "plot_history(history_wide, title='Modelo Largo (64 neurônios, 1 camada)')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Desafio 1 – Compare com o baseline\n",
        "\n",
        "Compare o **baseline** (16 neurônios) com o modelo **largo** (64 neurônios):\n",
        "\n",
        "- A **LOSS de treino** (azul) ficou menor com 64 neurônios?\n",
        "- E a **LOSS de validação** (vermelha)? Melhorou ou piorou?\n",
        "- O **gap** entre treino e validação aumentou ou diminuiu?\n",
        "\n",
        "Se o gap aumentou (treino bem melhor que validação), é um sinal de que a rede está com mais tendência a **overfitting**.\n",
        "\n",
        "Escreva em 1 frase o que você observou (opcional):"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "comentario_largo = \"\"\"Escreva aqui sua conclusão sobre o modelo largo vs baseline.\n",
        "\"\"\"\n",
        "print(comentario_largo)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5. Experimento 2 – Mais uma camada oculta (Profundidade)\n",
        "\n",
        "Agora vamos testar uma MLP com **duas camadas ocultas**:\n",
        "- 1ª camada: 32 neurônios (ReLU)\n",
        "- 2ª camada: 16 neurônios (ReLU)\n",
        "- Saída: 1 neurônio (sigmóide)\n",
        "\n",
        "Isso aumenta ainda mais a capacidade da rede – e também o risco de **overfitting**.\n",
        "\n",
        "Vamos treinar e observar as curvas de LOSS e accuracy."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def build_model_deeper(input_dim):\n",
        "    \"\"\"Modelo mais profundo (duas camadas ocultas).\"\"\"\n",
        "    model = keras.Sequential([\n",
        "        layers.Dense(32, activation='relu', input_shape=(input_dim,)),\n",
        "        layers.Dense(16, activation='relu'),\n",
        "        layers.Dense(1, activation='sigmoid')\n",
        "    ])\n",
        "    model.compile(\n",
        "        optimizer='adam',\n",
        "        loss='binary_crossentropy',\n",
        "        metrics=['accuracy']\n",
        "    )\n",
        "    return model\n",
        "\n",
        "deep_model = build_model_deeper(X_train.shape[1])\n",
        "\n",
        "history_deep = deep_model.fit(\n",
        "    X_train, y_train,\n",
        "    epochs=30,\n",
        "    batch_size=32,\n",
        "    validation_data=(X_val, y_val),\n",
        "    verbose=0\n",
        ")\n",
        "\n",
        "plot_history(history_deep, title='Modelo Profundo (2 camadas ocultas)')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Desafio 2 – Identificando overfitting\n",
        "\n",
        "Compare com o **baseline**:\n",
        "\n",
        "- As curvas de **LOSS** (treino/validação) ficaram melhores ou piores?\n",
        "- Você vê algum ponto (epoch) em que a **val_loss** começa a subir enquanto a loss de treino continua caindo?\n",
        "\n",
        "Se isso acontecer, é o **sinal clássico de overfitting**:\n",
        "> O modelo está aprendendo demais os detalhes do treino e piorando em dados novos.\n",
        "\n",
        "Anote sua observação (opcional):"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "comentario_profundo = \"\"\"Escreva aqui sua conclusão sobre o modelo profundo vs baseline.\n",
        "\"\"\"\n",
        "print(comentario_profundo)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6. Experimento 3 – Quantas epochs treinar?\n",
        "\n",
        "Vamos usar o **mesmo modelo baseline**, mas variar o número de epochs:\n",
        "- 10 epochs\n",
        "- 30 epochs\n",
        "- 100 epochs\n",
        "\n",
        "Ideia: ver a partir de qual ponto treinar mais **só piora a validação** (overfitting).\n",
        "\n",
        "> Repare principalmente na **val_loss**: se ela começa a subir, pode ser melhor parar o treino antes (técnica chamada *early stopping*)."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def train_with_epochs(epochs, title_prefix='Baseline'):\n",
        "    model = build_model_baseline(X_train.shape[1])\n",
        "    history = model.fit(\n",
        "        X_train, y_train,\n",
        "        epochs=epochs,\n",
        "        batch_size=32,\n",
        "        validation_data=(X_val, y_val),\n",
        "        verbose=0\n",
        "    )\n",
        "    plot_history(history, title=f'{title_prefix} ({epochs} epochs)')\n",
        "    return history\n",
        "\n",
        "history_10  = train_with_epochs(10,  'Baseline')\n",
        "history_30  = train_with_epochs(30,  'Baseline')\n",
        "history_100 = train_with_epochs(100, 'Baseline')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Desafio 3 – Encontrando o ponto “bom o suficiente”\n",
        "\n",
        "Observe principalmente o caso com **100 epochs**:\n",
        "\n",
        "- A **LOSS de treino** continua caindo quase sempre.\n",
        "- A **LOSS de validação** em algum momento começa a **subir**.\n",
        "\n",
        "Quando isso acontece?\n",
        "- Esse é o ponto em que o modelo começa a **decorar** o treino.\n",
        "- Na prática, poderíamos **parar o treino antes** (técnica de *early stopping*).\n",
        "\n",
        "Pergunta para refletir:\n",
        "- Se você tivesse limite de tempo de treino (por exemplo, só pode treinar por 20 epochs), qual configuração (número de neurônios, camadas etc.) você escolheria para ter um modelo **bom o suficiente**, comparando com o baseline?"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 7. Experimento 4 – Dropout para reduzir overfitting (opcional)\n",
        "\n",
        "Agora vamos adicionar **Dropout**, uma técnica simples de regularização.\n",
        "\n",
        "Ideia do Dropout:\n",
        "- Durante o treino, desligar aleatoriamente uma porcentagem de neurônios.\n",
        "- Isso força a rede a não depender demais de poucos neurônios específicos.\n",
        "- Na média, ajuda a reduzir **overfitting**.\n",
        "\n",
        "Vamos usar:\n",
        "- Duas camadas densas (64 e 32 neurônios).\n",
        "- Dropout de 50% entre elas.\n",
        "\n",
        "> Se estiver sem tempo em aula, esta parte pode virar um **desafio extra** ou atividade para casa."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "def build_model_dropout(input_dim):\n",
        "    \"\"\"Modelo com Dropout para reduzir overfitting.\"\"\"\n",
        "    model = keras.Sequential([\n",
        "        layers.Dense(64, activation='relu', input_shape=(input_dim,)),\n",
        "        layers.Dropout(0.5),\n",
        "        layers.Dense(32, activation='relu'),\n",
        "        layers.Dense(1, activation='sigmoid')\n",
        "    ])\n",
        "    model.compile(\n",
        "        optimizer='adam',\n",
        "        loss='binary_crossentropy',\n",
        "        metrics=['accuracy']\n",
        "    )\n",
        "    return model\n",
        "\n",
        "drop_model = build_model_dropout(X_train.shape[1])\n",
        "\n",
        "history_drop = drop_model.fit(\n",
        "    X_train, y_train,\n",
        "    epochs=30,\n",
        "    batch_size=32,\n",
        "    validation_data=(X_val, y_val),\n",
        "    verbose=0\n",
        ")\n",
        "\n",
        "plot_history(history_drop, title='Modelo com Dropout (64-32 neurônios)')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "### Desafio 4 – Dropout funcionou?\n",
        "\n",
        "Compare o modelo **com Dropout** com o modelo **largo sem Dropout** (64 neurônios):\n",
        "\n",
        "- A **LOSS de treino** com Dropout pode ficar um pouco pior (mais alta).\n",
        "- Mas a **LOSS de validação** melhorou? Ficou mais estável? O gap entre treino e validação diminuiu?\n",
        "\n",
        "Se sim, é sinal de que o Dropout ajudou a combater o **overfitting**.\n",
        "\n",
        "> Moral da história: às vezes, aceitar um treino um pouco pior (LOSS maior no treino) pode trazer um desempenho **melhor em dados novos**.\n",
        "\n",
        "Anote sua observação (opcional):"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "comentario_dropout = \"\"\"Escreva aqui sua conclusão sobre o efeito do Dropout.\n",
        "\"\"\"\n",
        "print(comentario_dropout)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 8. Avaliando no conjunto de teste\n",
        "\n",
        "Vamos escolher **um modelo** (por exemplo, o com Dropout) e avaliar no conjunto de teste, que o modelo nunca viu.\n",
        "\n",
        "Aqui vou usar o **modelo com Dropout**, mas você pode repetir com qualquer um dos outros modelos e comparar.\n",
        "\n",
        "> Lembre: o desempenho no **teste** é o que mais se aproxima do que aconteceria em produção (dados realmente novos)."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "execution_count": null,
      "outputs": [],
      "source": [
        "test_loss, test_acc = drop_model.evaluate(X_test, y_test, verbose=0)\n",
        "print(f'LOSS no teste: {test_loss:.4f}')\n",
        "print(f'Accuracy no teste: {test_acc:.4f}')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 9. Resumo da Aula 5\n",
        "\n",
        "Hoje você:\n",
        "\n",
        "- Viu que **LOSS** é a medida principal de erro que o modelo tenta **minimizar** durante o treinamento.\n",
        "- Aprendeu a olhar **LOSS de treino x LOSS de validação** para identificar:\n",
        "  - **Underfitting**: LOSS alta em tudo, modelo não aprende o padrão.\n",
        "  - **Overfitting**: treino melhora (LOSS caindo), mas validação piora (LOSS subindo).\n",
        "- Definiu e usou um **modelo baseline** (MLP simples) como referência.\n",
        "- Brincou com a arquitetura da MLP:\n",
        "  - Número de neurônios (largura).\n",
        "  - Número de camadas (profundidade).\n",
        "  - Número de epochs (tempo de treino).\n",
        "  - Dropout (regularização para reduzir overfitting).\n",
        "\n",
        "### Perguntas para reflexão\n",
        "\n",
        "1. Como você explicaria **LOSS** em **uma frase** para alguém da área de negócios?\n",
        "2. Se o seu modelo tem **val_loss** muito maior que **loss**, o que você tentaria fazer para melhorar?\n",
        "3. O modelo com mais neurônios ou mais camadas foi realmente **melhor** que o baseline? Em que aspecto?\n",
        "\n",
        "### Próxima aula\n",
        "\n",
        "Na próxima aula, vamos abrir a caixa-preta:\n",
        "- Como o modelo realmente **ajusta os pesos** para reduzir a LOSS?\n",
        "- Intuição de **Gradiente Descendente** e **Backpropagation**, conectando com os slides de conceitos fundamentais vistos no início da disciplina."
      ]
    }
  ]
}